# Gateway tracking with an uncertain halo observer

Compare Gateway orbit estimation over one observer period using (1) exact observer knowledge and (2) an observer orbit estimated from lunar surface observations. Both cases use the same truth trajectories, Gateway prior and noisy Gateway measurements.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "configs/baseline.json").exists() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the project checkout")
sys.path.insert(0, str(ROOT / "src"))
from halo_tracking.study import load_config
cfg = load_config(ROOT / "configs/baseline.json")
cfg["simulation"]["periods"] = 1.0  # One nominal observer period


## 1. Matched comparison

| Assumption | Perfect observer | LSO-estimated observer |
|---|---|---|
| Observer state supplied to Gateway estimator | Exact true state | State estimated from LSO angles |
| Initial observer uncertainty | Zero | 1 km and 0.01 m/s per axis, 1σ |
| Initial Gateway uncertainty | 1 km and 0.01 m/s per axis, 1σ | Same |
| Observer-to-Gateway camera | 1 arcsecond per angle | Same measurements and noise |
| LSO | Not needed | 30° N at 270° E and 90° E; 85° S at 90° E; 1 arcsecond per angle |

These are the baseline settings; the configuration table below displays the values actually used. Both optical links use **angles only** in this focused comparison. The cadence is 30 minutes. Earth/Moon occultation is included for Gateway observations; the LSO uses the configured elevation mask. Optical brightness, Sun exclusions and camera attitude errors are not modeled.

The true observer remains on the supplied initial-value trajectory in both cases. The imperfect case introduces uncertainty in our **knowledge** of it, not a different physical orbit.

In [ ]:
from IPython.display import HTML, Markdown, display
settings = [
    ("Duration", f"1 observer nominal period = {cfg['orbit']['period_tu'] * cfg['time_unit_s'] / 86400:.2f} days"),
    ("Observer initial position / velocity 1σ", f"{cfg['estimation']['initial_position_sigma_km']:g} km / {1000*cfg['estimation']['initial_velocity_sigma_km_s']:g} m/s per axis"),
    ("Gateway initial position / velocity 1σ", f"{cfg['comparison']['gateway_position_sigma_km']:g} km / {1000*cfg['comparison']['gateway_velocity_sigma_km_s']:g} m/s per axis"),
    ("LSO angular noise 1σ", f"{cfg['measurement']['angle_sigma_arcsec']:g} arcsec per angle"),
    ("Gateway camera angular noise 1σ", f"{cfg['comparison']['gateway_camera_sigma_arcsec']:g} arcsec per angle")]
display(HTML("<table>"+"".join(f"<tr><th>{name}</th><td>{value}</td></tr>" for name,value in settings)+"</table>"))

In [ ]:
from halo_tracking.gateway_comparison import run_gateway_comparison, save_comparison
result = run_gateway_comparison(cfg)
report = save_comparison(result, ROOT / "results/gateway_comparison")

## 2. Geometry and what the LSO knows

The first figure spans exactly one observer nominal period (Gateway repeats at its own period). It is context: both true trajectories are identical across the two estimation cases. The dots mark their initial states. The axis scale is in thousands of kilometers, with equal physical scale on all three axes.

The second figure shows the observer's estimated-state error and its local covariance prediction. The shaded intervals contain observations from at least one LSO. The three-site network is evaluated at every scheduled epoch. **RSS σ is a covariance summary, not a guaranteed bound on the error magnitude.**

In [ ]:
from halo_tracking.plotting import geometry, observer_knowledge, gateway_comparison_figures, state_histories, export
fig_geometry = geometry(result)
export(fig_geometry, ROOT / "results/gateway_comparison/geometry.png")

In [ ]:
fig_observer = observer_knowledge(result)
export(fig_observer, ROOT / "results/gateway_comparison/observer.png")

## Observer position, velocity and acceleration

The solid curve is the perfect observer state (truth); the dashed curve is the LSO-predicted state. Components use Moon-centered inertial axes. These curves nearly overlap at orbit scale; the error plot above resolves their differences.

In [ ]:
fig_observer_states = state_histories(result, 'observer')
export(fig_observer_states, ROOT / 'results/gateway_comparison/observer_states.png')

## Gateway position, velocity and acceleration

Compare the true Gateway trajectory with the two tracked trajectories: one uses the perfect observer and one uses the LSO-predicted observer. Their small separation is quantified by the error plot and RMSE table below.

In [ ]:
fig_gateway_states = state_histories(result, 'gateway')
export(fig_gateway_states, ROOT / 'results/gateway_comparison/gateway_states.png')

## 3. What changes in Gateway tracking

**Actual-error plot:** distance between the estimated and true Gateway state in this one shared noise realization. Lower is better. It is an actual orbit estimate, not just a pointing-error projection.

**Predicted-uncertainty plot:** square root of the position/velocity covariance-block trace. Lower means the estimator predicts better knowledge. These are local linearized predictions; they can be optimistic if the approximation is inaccurate.

In the imperfect case, the estimator carries observer–Gateway cross-covariance through time. Gateway measurements do not refine the observer: its state knowledge comes from the LSO. LSO updates can also correct Gateway indirectly through the retained correlation.

In [ ]:
figures = gateway_comparison_figures(result)
fig_errors = figures["errors"]
export(fig_errors, ROOT / "results/gateway_comparison/errors.png")
import matplotlib.pyplot as plt
plt.close(figures["uncertainty"])

In [ ]:
figures_uncertainty = gateway_comparison_figures(result)
plt.close(figures_uncertainty["errors"])
fig_uncertainty = figures_uncertainty["uncertainty"]
export(fig_uncertainty, ROOT / "results/gateway_comparison/uncertainty.png")

## 4. Numerical conclusions

The table separates **actual errors from one realization** from **predicted uncertainty**. It should be read with the assumptions above, especially the Gateway prior and observing geometry.

In [ ]:
rows=[]
for key,label in [("perfect","Perfect observer knowledge"),("lso_estimated","Observer estimated from LSO")]:
    item=report[key]
    values=[label, f"{1000*item['position_rmse_km']:.1f}", f"{item['velocity_rmse_m_s']:.4f}",
            f"{1000*item['final_position_error_km']:.1f}",
            f"{1000*item['final_position_forecast_rss_sigma_km']:.1f}",
            f"{item['final_velocity_error_m_s']:.3g}"]
    rows.append("<tr>"+"".join("<td>"+v+"</td>" for v in values)+"</tr>")
headers=["Observer knowledge", "Gateway position RMSE over arc (m)", "Gateway velocity RMSE over arc (m/s)", "Final actual position error (m)", "Final predicted position RSS σ (m)", "Final actual velocity error (m/s)"]
display(HTML("<table><tr>"+"".join("<th>"+v+"</th>" for v in headers)+"</tr>"+"".join(rows)+"</table>"))

In [ ]:
import numpy as np
ideal=report["perfect"]; estimated=report["lso_estimated"]
ratio=estimated["final_position_forecast_rss_sigma_km"]/ideal["final_position_forecast_rss_sigma_km"]
available=np.flatnonzero(result["lso_available"])
start=result["times"][available[0]]*cfg["time_unit_s"]/86400 if len(available) else float("nan")
end=result["times"][available[-1]]*cfg["time_unit_s"]/86400 if len(available) else float("nan")
conclusions = f"""
- Final predicted Gateway position RSS σ: **{1000*ideal['final_position_forecast_rss_sigma_km']:.1f} m** with exact observer knowledge versus **{1000*estimated['final_position_forecast_rss_sigma_km']:.1f} m** with LSO-estimated knowledge ({ratio:.1f}×).
- Final actual position errors: **{1000*ideal['final_position_error_km']:.1f} m** versus **{1000*estimated['final_position_error_km']:.1f} m** in this realization.
- The network supplies **{int(result['sensor_visible'].sum())} site observations** across **{int(result['lso_available'].sum())} scheduled epochs**. Overlapping sites provide separate measurements.
- These EKF results illustrate ideal versus measured state knowledge. They are a benchmark, not an identified model or an ensemble accuracy claim.
"""
display(Markdown(conclusions))

## Model and limitations

Propagation uses the Earth–Moon CR3BP. The plots transform normalized rotating states into Moon-centered inertial position, velocity and acceleration. Acceleration is calculated from each state using the dynamics; it is not independently estimated.

The imperfect case retains observer–Gateway cross-covariance. LSO measurements update observer knowledge; Gateway measurements update Gateway without refining the observer. Initial covariances are diagonal, with zero observer–Gateway cross-covariance. The forecast uses zero innovations along the nominal trajectories; the actual estimate uses noisy measurements.

The supplied `SNRHL2` state is a southern NRHO initial condition rounded to four decimals. Its one-period closure miss is about 99 km. This study assumes no stationkeeping or unmodeled forces. Local filter covariance can be overconfident near nonlinear updates, so predicted uncertainty is a conditional comparison rather than a validated error bound.

Gateway's initial state comes from [the earlier cislunar project](https://github.com/scottblender/Gradient-Free-Optimization-for-Cislunar-SDA-2/blob/main/scripts/build_target_case_database.m). Both phases start at model time zero, without a mission epoch. Both trajectories use LU = 384400 km and TU = 375190.25852 s; the source Gateway project used TU = 375695 s.

## Uncontrolled departure over 20 periods

Use the supplied CR3BP trajectory over **one nominal period** as requested, then repeat its states at the same rotating-frame orbital phase. Compare that reference with a continuous, uncontrolled 20-period propagation from the same initial state. The Gateway/EKF comparison above spans one observer period.

The original rounded catalog reference does not close after one nominal period. Resetting its reference trajectory creates jumps in the departure diagnostic; the actual uncontrolled trajectory is propagated continuously and diverges from that nominal reference. The instantaneous departure curves therefore contain reference-reset jumps. Equal-phase period RMS averages the same 500 phase samples per period, excluding each right boundary; it makes the sustained trend easier to see but still includes accumulated timing/phase drift.

This is departure from the chosen reference, not proof of chaos or an onset-of-instability test. No maneuver deadline is selected yet. The first traversal matches the reference and is omitted from the logarithmic plots.

In [ ]:
from halo_tracking.departure import reference_departure, save_departure
from halo_tracking.plotting import departure_figures
departure = reference_departure(cfg, periods=20, ensemble_count=8)
save_departure(departure, cfg, ROOT / 'results/departure')
departure_plots = departure_figures(departure, cfg)
export(departure_plots['departure'], ROOT / 'results/departure/departure.png')
export(departure_plots['period_growth'], ROOT / 'results/departure/period_growth.png')

In [ ]:
closure = departure['diagnostics']
periods = [2, 4, 6, 8, 10, 20]
rows = ''.join(f"<tr><td>{k}</td><td>{departure['period_position_rms_km'][k-1]:,.1f}</td>"
               f"<td>{departure['period_velocity_rms_m_s'][k-1]:,.3f}</td></tr>" for k in periods)
display(HTML('<table><tr><th>Period</th><th>RMS position departure (km)</th>'
             '<th>RMS velocity departure (m/s)</th></tr>' + rows + '</table>'))
display(Markdown(f"""
The one-period reference closure mismatch is **{closure['position_closure_km']:.1f} km** and **{closure['velocity_closure_m_s']:.3f} m/s**.

Rapid divergence is evaluated from this particular rounded-reference simulation and the 1 km / 0.01 m/s initial perturbation ensemble. Do not interpret transient decreases as stabilization, or set an operational cutoff before defining position/velocity tolerances.

The next stationkeeping decision is an acceptable departure corridor and prediction horizon. These curves support choosing that requirement; they do not establish the latest safe maneuver time by themselves.
"""))

## Cumulative uncontrolled observer trajectories

Every panel starts at the **same** rounded SNRHL2 initial state and displays the **continuous** trajectory over `0 → kT` (not the isolated kth period), for `k=1,...,20`. The dashed curve shows the original one-period nominal reference; it is **not** a second propagated spacecraft. The original reference has a ~99-km closure error, so departure from it is a nominal-reference diagnostic rather than a validated orbit-stability boundary.

The first grid auto-scales each panel to reveal the entire growing path. The second keeps the reference-orbit axes fixed, intentionally clipping trajectories that escape far beyond the nominal region. Each panel labels the maximum reference departure **any time up to its horizon**, not the actual range to Gateway.

In [ ]:
from halo_tracking.plotting import period_trajectory_grid
fig_period_grid = period_trajectory_grid(departure, cfg, scale_mode='local')
export(fig_period_grid, ROOT / 'results/departure/observer_period_grid.png')
display(fig_period_grid)
plt.close(fig_period_grid)

fig_reference_grid = period_trajectory_grid(departure, cfg, scale_mode='reference')
export(fig_reference_grid, ROOT / 'results/departure/observer_period_grid_fixed_scale.png')
plt.close(fig_reference_grid)

## Gateway tracking versus cumulative observer-period horizon

The Gateway study performs **one continuous simulation and estimator run**, starting from the original rounded observer initial condition. For each `k`, the row summarizes **all samples from `0 → kT`**, not just the kth period. There are no filter or covariance resets.

**Interpret these distances separately:**
- **Observer departure** is the observer's difference from the nominal one-period halo reference at matching phase. It is not distance traveled, distance from the Moon, or range to Gateway. The rounded reference does not close perfectly, so this is a useful but imperfect orbital-departure diagnostic.
- **Observer–Gateway range** is the actual Euclidean separation between the two spacecraft, relevant to the optical detection-range cutoff.

The table reports cumulative position/velocity RMSE, cumulative accepted Gateway and lunar-surface observations, **new observations in the most recent period**, and terminal predictive position uncertainty. If new observations are zero, the estimate is simply propagating its prior model; low cumulative RMSE does **not** mean Gateway is currently observable.

Measurements are rejected using geometric occultation, the configured 20° Sun / 10° Earth center-referenced keep-outs, and illustrative 250,000-km range caps. The Sun direction is a simplified model, not a calibrated ephemeris or instrument detection model.

Saved CSV rows are reused only if the configuration, requested horizon, and **cumulative-metrics version** all match; prior per-period files are invalidated.

In [ ]:
import os
from halo_tracking.gateway_comparison import (
    run_gateway_period_study, save_gateway_period_study, load_gateway_period_study
)

observer_period_count = int(os.environ.get('HALO_GATEWAY_PERIODS', '20'))
period_output = ROOT / 'results/departure'
gateway_period_study = load_gateway_period_study(
    cfg, period_output, periods=observer_period_count)
if gateway_period_study is None:
    gateway_period_study = run_gateway_period_study(
        cfg, periods=observer_period_count)
    save_gateway_period_study(gateway_period_study, period_output)
    print('Computed cumulative 0→kT Gateway metrics and saved CSV.')
else:
    print('Reused cumulative Gateway metrics CSV:', period_output / 'gateway_period_metrics.csv')

period_rows = gateway_period_study['rows']
columns = [
    ('horizon_periods', 'Horizon', lambda v: f'0 → {v}T'),
    ('case', 'Observer knowledge', str),
    ('terminal_observer_departure_km', 'Endpoint orbit departure (km)', lambda v: f'{v:,.1f}'),
    ('terminal_gateway_range_km', 'Endpoint Gateway range (km)', lambda v: f'{v:,.0f}'),
    ('gateway_observations', 'Gateway obs., cumulative', str),
    ('lso_observations', 'LSO obs., cumulative', str),
    ('new_gateway_observations', 'Gateway obs., last T', str),
    ('new_lso_observations', 'LSO obs., last T', str),
    ('position_rmse_km', 'Gateway position RMSE 0→kT (km)', lambda v: f'{v:,.3f}'),
    ('velocity_rmse_m_s', 'Gateway velocity RMSE 0→kT (m/s)', lambda v: f'{v:,.4f}'),
    ('final_position_forecast_rss_sigma_km', 'Endpoint predicted RSS σ (km)', lambda v: f'{v:,.3f}'),
]
heading = ''.join(f'<th style="position:sticky;top:0;background:#f2f5f8;'
                  f'border-bottom:1px solid #cdd5df;padding:9px 10px;'
                  f'white-space:normal;">{label}</th>' for _, label, _ in columns)
body = ''.join(
    '<tr style="background:' + ('#f7f9fc' if (i//2)%2 else 'white') + '">'
    + ''.join(f'<td style="padding:7px 10px;border-bottom:1px solid #e6eaf0;'
              f'text-align:{"left" if key=="case" else "right"};">'
              f'{format_value(item[key])}</td>'
              for key, _, format_value in columns)
    + '</tr>'
    for i, item in enumerate(period_rows)
)
display(HTML(
    '<div style="max-width:100%;overflow-x:auto;max-height:650px;'
    'border:1px solid #dce3eb;border-radius:8px;">'
    '<table style="border-collapse:collapse;font:12px system-ui,sans-serif;'
    'font-variant-numeric:tabular-nums;min-width:1320px;width:100%;">'
    '<thead><tr>' + heading + '</tr></thead><tbody>' + body
    + '</tbody></table></div>'
))

## System identification — step 1: generate data

**Goal:** learn a local, unforced state-transition model that predicts the observer's next position and velocity. No model is fitted in this step.

Generate 20 training, 5 validation and 5 held-out test arcs. Each spans one nominal period, sampled uniformly every 30 minutes. Initial states are independently perturbed about the supplied state with 1 km and 0.01 m/s standard deviation per rotating-frame axis. These describe the local training domain, separately from the EKF's estimation prior.

Each file contains normalized rotating truth states, physical inertial position/velocity/acceleration, ideal angles, noisy angles and per-site visibility. Measured angles are NaN when unavailable. Training targets are the next normalized six-component state; the current state is the input. Truth states are simulation labels, not LSO position measurements.

Split by whole trajectory before constructing lag windows. Fit normalization using training data only; use validation for model selection and test data once for final assessment. Do not use invisible ideal angles as observations. The current dataset covers nearby initial conditions at one nominal phase, with no thrust or model discrepancy; it does not yet support stationkeeping identification.

In [ ]:
from halo_tracking.identification import generate_dataset
identification_summary = generate_dataset(cfg, ROOT / 'results/system_identification')
headers = ['Split', 'Arcs', 'Samples / arc', 'Transitions', 'Network visibility']
rows = ''.join('<tr>' + ''.join(f'<td>{value}</td>' for value in
    [name, item['arcs'], item['samples_per_arc'], item['transitions'], f"{100*item['visible_fraction']:.1f}%"])
    + '</tr>' for name, item in identification_summary.items())
display(HTML('<table><tr>' + ''.join(f'<th>{h}</th>' for h in headers) + '</tr>' + rows + '</table>'))